# 02 — First cost model results: how long can a driver queue?

**Question:** at what queue time does CNG conversion stop paying off for a ride-hailing driver and a keke rider?

This notebook runs the daily cost model in `src/cost_model.py` for two vehicles, three input scenarios and a range of
driver earnings, then finds the **break-even queue time**: the queue length at which the fuel saving is used up by
lost earnings and loan repayments.

All inputs come from `data/raw/model_inputs.csv` (sources in `docs/sources.md`). Values that are **assumptions** or
**derived** rather than sourced are labelled as such wherever they appear.

## 1. Setup

Same pattern as notebook 01: find the project root, add `src/` to the import path, then import our own modules.
`cost_model` is imported as `cm` so calls read as `cm.fuel_saving_per_day(...)`, which makes it obvious where each
function lives.

In [1]:
import sys
from pathlib import Path

import pandas as pd
import plotly.graph_objects as go

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT / "src"))
import load_data          # noqa: E402
import cost_model as cm   # noqa: E402

FIG_DIR = PROJECT_ROOT / "outputs" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

# Show money without scientific notation in tables
pd.options.display.float_format = "{:,.2f}".format

## 2. Load the inputs

`get_input` looks up one row of `model_inputs.csv` by parameter and vehicle and returns its low / mid / high values
as numbers, plus the source IDs. It raises an error if the row is missing or duplicated, so a typo in a parameter
name fails loudly instead of quietly producing a wrong answer. Blank cells become `None`.

In [2]:
inputs = load_data.read_csv_safe(load_data.RAW_DIR / "model_inputs.csv")
LEVELS = ["low", "mid", "high"]


def get_input(parameter, vehicle):
    """Return {'low': x, 'mid': y, 'high': z, 'source': 'S..'} for one row of model_inputs.csv."""
    match = inputs[(inputs["parameter"] == parameter) & (inputs["vehicle"] == vehicle)]
    if len(match) != 1:
        raise ValueError(f"Expected 1 row for {parameter} / {vehicle}, found {len(match)}")
    row = match.iloc[0]
    values = {level: float(row[level]) if row[level].strip() else None for level in LEVELS}
    values["source"] = row["source_ids"] or row["confidence"]   # e.g. 'S18', or 'Assumption'
    return values


petrol_price    = get_input("petrol_price_national", "all")   # NGN per litre
cng_price       = get_input("cng_price", "car/bus")           # NGN per scm
interest_pct    = get_input("loan_interest_rate", "all")      # percent per year
tenor           = get_input("loan_tenor", "all")              # years
queue_time      = get_input("queue_time_per_refuel", "all")   # hours per refuel
working_days    = get_input("working_days_per_year", "all")   # days per year (Assumption)
conv_car        = get_input("conversion_cost", "car")         # NGN
conv_keke       = get_input("conversion_cost", "keke")        # NGN
spend_car       = get_input("ridehail_daily_fuel_petrol", "car")    # NGN per day, mid only
spend_keke      = get_input("keke_daily_fuel_cost_petrol", "keke")  # NGN per day, mid only

### A gap to fill: keke conversion cost has no mid value

The keke row gives only a low (₦100,000, official estimate, S43) and a high (₦650,000, one reported quote, S04).
For the mid scenario we use the **midpoint, ₦375,000**. This is a **derived** number, not a published one; the driver
survey should replace it. We fill it in code (not in the CSV) so `data/raw/` stays as published.

The keke also uses the `car/bus` CNG price, since that is the only light-vehicle price in the file.

In [3]:
conv_keke["mid"] = (conv_keke["low"] + conv_keke["high"]) / 2
conv_keke["source"] += " (mid = derived midpoint)"
conv_keke

{'low': 100000.0,
 'mid': 375000.0,
 'high': 650000.0,
 'source': 'S43;S04 (mid = derived midpoint)'}

## 3. Vehicle profiles and the petrol-spend rescaling

Both daily petrol spends were reported when petrol cost **about ₦1,000 per litre** (keke: S19, 2024; car: S21,
pre-2026). Petrol is much more expensive in every scenario now, so we rescale:

> rescaled spend = reported spend × scenario petrol price ÷ 1,000

**This assumes each driver buys the same number of litres per day as when the spend was reported**, i.e. covers the
same distance. If drivers have cut back on driving as prices rose, this overstates today's spend and the fuel saving.
Distance per day is a survey item (`daily_distance` is a Gap row in `model_inputs.csv`).

In [4]:
SPEND_REFERENCE_PETROL_PRICE = 1_000   # NGN/litre when the spends were reported (S19 note; S21 pre-2026)

vehicles = {
    "Ride-hailing car": {
        "spend_reported": spend_car["mid"],     # 20,000 NGN/day, S21
        "spend_source": spend_car["source"],
        "conversion_cost": conv_car,
    },
    "Keke": {
        "spend_reported": spend_keke["mid"],    # 15,000 NGN/day, S19
        "spend_source": spend_keke["source"],
        "conversion_cost": conv_keke,
    },
}

pd.DataFrame({
    name: {
        "Reported petrol spend (NGN/day)": v["spend_reported"],
        "Spend source": v["spend_source"],
        "Conversion cost low (NGN)": v["conversion_cost"]["low"],
        "Conversion cost mid (NGN)": v["conversion_cost"]["mid"],
        "Conversion cost high (NGN)": v["conversion_cost"]["high"],
        "Conversion cost source": v["conversion_cost"]["source"],
    }
    for name, v in vehicles.items()
})

,Ride-hailing car,Keke
Reported petrol spend (NGN/day),"20,000.00","15,000.00"
Spend source,S21,S19
Conversion cost low (NGN),"300,000.00","100,000.00"
Conversion cost mid (NGN),"1,300,000.00","375,000.00"
Conversion cost high (NGN),"1,700,000.00","650,000.00"
Conversion cost source,S42;S40;S41,S43;S04 (mid = derived midpoint)


## 4. Scenarios

Each scenario takes the **same-named column** of `model_inputs.csv` for every input: "low" uses the low petrol price,
low CNG price, low conversion cost, low interest rate, shortest tenor and shortest queue. So "low" is *not* a best or
worst case. Some low values help CNG (cheap CNG, cheap kit) and some hurt it (cheap petrol, a 1-year loan with bigger
daily repayments).

Two more **assumptions**, the same in every scenario:
- **1 refuel per day.** A car tank holds about 200 km of CNG (S20), so one fill a day is plausible for a full-time driver;
  the survey asks directly.
- **efficiency_ratio = 1.0**, meaning 1 scm of CNG goes as far as 1 litre of petrol (conflict C4, S20 and S05). This
  calibration is for cars; we apply it to keke too until survey data says otherwise.

In [5]:
REFUELS_PER_DAY = 1        # Assumption: pending survey
EFFICIENCY_RATIO = 1.0     # Assumption: 1 scm ~ 1 litre on distance (C4)

scenarios = {
    level: {
        "petrol_price": petrol_price[level],
        "cng_price": cng_price[level],
        "interest_rate": interest_pct[level] / 100,   # cost_model wants a decimal: 17.5% -> 0.175
        "tenor_years": tenor[level],
        "queue_hours": queue_time[level],
    }
    for level in LEVELS
}

# For display only: show the interest rate as a percentage (0.175 would print as 0.17 at 2 decimals)
display_scenarios = pd.DataFrame(scenarios)
display_scenarios.loc["interest_rate"] = display_scenarios.loc["interest_rate"] * 100
display_scenarios.rename(index={"interest_rate": "interest_rate_pct"})

,low,mid,high
petrol_price,"1,051.00","1,400.00","1,596.00"
cng_price,230.00,318.00,380.00
interest_rate_pct,15.00,17.50,20.00
tenor_years,1.00,2.00,3.00
queue_hours,1.50,4.00,6.00


## 5. Earnings per hour: a sensitivity range

Net earnings per hour (after owner remittance and app commission) are **not yet known**; this is a survey item. Rather
than guess one number, we test **₦1,000 to ₦5,000 per hour in steps of ₦500**.

> ⚠️ This is a **sensitivity range pending survey data**, not an estimate of what drivers earn.

Earnings matter because they set the price of an hour in the queue: the more a driver earns, the more each hour of
waiting costs them, and the shorter the queue they can afford.

In [6]:
EARNINGS_PER_HOUR = list(range(1_000, 5_001, 500))   # NGN/hour; range() stops before 5,001, so 5,000 is included
EARNINGS_PER_HOUR

[1000, 1500, 2000, 2500, 3000, 3500, 4000, 4500, 5000]

## 6. Run the model

One loop over every combination: 2 vehicles × 3 scenarios × 9 earnings levels = 54 rows. Each row calls the
`cost_model` functions in the order the daily budget works: fuel saving, then queue cost, loan cost, net benefit, and
finally the break-even queue time.

We collect plain dictionaries in a list and build the DataFrame once at the end, which is simpler and faster than
adding rows to a DataFrame one at a time.

In [7]:
rows = []
for vehicle_name, v in vehicles.items():
    for level, s in scenarios.items():
        # Rescale the reported spend to this scenario's petrol price (same litres per day)
        petrol_spend = v["spend_reported"] * s["petrol_price"] / SPEND_REFERENCE_PETROL_PRICE
        conversion_cost = v["conversion_cost"][level]

        # These two don't depend on earnings, so work them out once per scenario
        fuel_saving = cm.fuel_saving_per_day(petrol_spend, s["petrol_price"], s["cng_price"], EFFICIENCY_RATIO)
        loan_cost = cm.loan_cost_per_day(conversion_cost, s["interest_rate"], s["tenor_years"], working_days[level])

        for earnings in EARNINGS_PER_HOUR:
            queue_cost = cm.queue_cost_per_day(REFUELS_PER_DAY, s["queue_hours"], earnings)
            rows.append({
                "vehicle": vehicle_name,
                "scenario": level,
                "earnings_per_hour": earnings,
                "petrol_spend_per_day": petrol_spend,
                "fuel_saving": fuel_saving,
                "queue_cost": queue_cost,
                "loan_cost": loan_cost,
                "net_benefit": cm.net_benefit_per_day(fuel_saving, queue_cost, loan_cost),
                "break_even_queue_hours": cm.break_even_queue_hours(
                    fuel_saving, loan_cost, REFUELS_PER_DAY, earnings),
            })

results = pd.DataFrame(rows)
print(f"{len(results)} rows")

54 rows


## 7. Mid-scenario summary

All money columns are **naira per working day**; break-even is **hours per refuel**. Queue cost uses the mid queue
time of 4 hours. Break-even does *not* depend on the actual queue time: it is the queue time that would make net
benefit zero.

In [8]:
mid = results[results["scenario"] == "mid"].drop(columns="scenario")

mid.rename(columns={
    "vehicle": "Vehicle",
    "earnings_per_hour": "Earnings (NGN/hr)",
    "petrol_spend_per_day": "Petrol spend (NGN/day)",
    "fuel_saving": "Fuel saving (NGN/day)",
    "queue_cost": "Queue cost (NGN/day)",
    "loan_cost": "Loan cost (NGN/day)",
    "net_benefit": "Net benefit (NGN/day)",
    "break_even_queue_hours": "Break-even queue (hrs)",
}).set_index(["Vehicle", "Earnings (NGN/hr)"])

Petrol spend (NGN/day)  \
Vehicle          Earnings (NGN/hr)                           
Ride-hailing car 1000                            28,000.00   
                 1500                            28,000.00   
                 2000                            28,000.00   
                 2500                            28,000.00   
                 3000                            28,000.00   
                 3500                            28,000.00   
                 4000                            28,000.00   
                 4500                            28,000.00   
                 5000                            28,000.00   
Keke             1000                            21,000.00   
                 1500                            21,000.00   
                 2000                            21,000.00   
                 2500                            21,000.00   
                 3000                            21,000.00   
                 3500                            21,000.00   
                 4000                            21,000.00   
                 4500                            21,000.00   
                 5000                            21,000.00   

                                    Fuel saving (NGN/day)  \
Vehicle          Earnings (NGN/hr)                          
Ride-hailing car 1000                           21,640.00   
                 1500                           21,640.00   
                 2000                           21,640.00   
                 2500                           21,640.00   
                 3000                           21,640.00   
                 3500                           21,640.00   
                 4000                           21,640.00   
                 4500                           21,640.00   
                 5000                           21,640.00   
Keke             1000                           16,230.00   
                 1500                           16,230.00   
                 2000                           16,230.00   
                 2500                           16,230.00   
                 3000                           16,230.00   
                 3500                           16,230.00   
                 4000                           16,230.00   
                 4500                           16,230.00   
                 5000                           16,230.00   

                                    Queue cost (NGN/day)  Loan cost (NGN/day)  \
Vehicle          Earnings (NGN/hr)                                              
Ride-hailing car 1000                           4,000.00             2,484.14   
                 1500                           6,000.00             2,484.14   
                 2000                           8,000.00             2,484.14   
                 2500                          10,000.00             2,484.14   
                 3000                          12,000.00             2,484.14   
                 3500                          14,000.00             2,484.14   
                 4000                          16,000.00             2,484.14   
                 4500                          18,000.00             2,484.14   
                 5000                          20,000.00             2,484.14   
Keke             1000                           4,000.00               716.58   
                 1500                           6,000.00               716.58   
                 2000                           8,000.00               716.58   
                 2500                          10,000.00               716.58   
                 3000                          12,000.00               716.58   
                 3500                          14,000.00               716.58   
                 4000                          16,000.00               716.58   
                 4500                          18,000.00               716.58   
                 5000              

### Break-even queue time across all three scenarios

The chart below shows only the mid scenario, so this table shows how much the answer moves when every input shifts to
its low or high value. Hours per refuel.

In [9]:
results.pivot_table(
    index="earnings_per_hour",
    columns=["vehicle", "scenario"],
    values="break_even_queue_hours",
)[[(v, s) for v in vehicles for s in LEVELS]]   # keep low / mid / high order instead of alphabetical

vehicle           Ride-hailing car              Keke            
scenario                       low   mid  high   low   mid  high
earnings_per_hour                                               
1000                         15.38 19.16 21.89 11.97 15.51 17.31
1500                         10.25 12.77 14.59  7.98 10.34 11.54
2000                          7.69  9.58 10.95  5.98  7.76  8.66
2500                          6.15  7.66  8.76  4.79  6.21  6.92
3000                          5.13  6.39  7.30  3.99  5.17  5.77
3500                          4.39  5.47  6.25  3.42  4.43  4.95
4000                          3.84  4.79  5.47  2.99  3.88  4.33
4500                          3.42  4.26  4.86  2.66  3.45  3.85
5000                          3.08  3.83  4.38  2.39  3.10  3.46

## 8. Chart: break-even queue time vs earnings

- **One line per vehicle, mid scenario.** Blue = ride-hailing car, orange = keke, the same palette as notebook 01.
- **Grey band = reported queue times**, 1.5 hours (NESG estimate, S02) to 6 hours (Abuja reports, S04), read from the
  `queue_time_per_refuel` row. Where a line sits *above* the band, CNG pays off even at the worst reported queues;
  where it drops *into* the band, real queues can wipe out the gain.
- `add_hrect` draws the band as a shape spanning the full width, with `layer="below"` so the lines sit on top.
- **y-axis starts at 0** so the distance between a line and the band isn't distorted.

In [10]:
COLORS = {"Ride-hailing car": "#2a78d6", "Keke": "#eb6834"}
LABEL_SHIFT = {"Ride-hailing car": 8, "Keke": -8}   # pixels
INK = "#0b0b0b"
INK_MUTED = "#52514e"
BAND_LOW, BAND_HIGH = queue_time["low"], queue_time["high"]   # 1.5 and 6 hours (S02; S04)
m = scenarios["mid"]   # short name for the subtitle and footnote

fig = go.Figure()

fig.add_hrect(
    y0=BAND_LOW, y1=BAND_HIGH,
    fillcolor="#8a8984", opacity=0.15, line_width=0, layer="below",
    annotation_text=f"Reported queue times (S02, S04): {BAND_LOW:g}–{BAND_HIGH:g} hrs",
    annotation_position="top left",
    annotation_font=dict(color=INK_MUTED, size=12),
)

for vehicle_name in vehicles:
    d = mid[mid["vehicle"] == vehicle_name]
    fig.add_trace(go.Scatter(
        x=d["earnings_per_hour"],
        y=d["break_even_queue_hours"],
        name=vehicle_name,
        mode="lines+markers",
        line=dict(color=COLORS[vehicle_name], width=2),
        marker=dict(size=8, color=COLORS[vehicle_name], line=dict(color="white", width=2)),
        customdata=d[["fuel_saving", "loan_cost"]],
        hovertemplate=(
            f"<b>{vehicle_name}</b><br>"
            "Earnings: ₦%{x:,.0f} per hour<br>"
            "Break-even queue: <b>%{y:.1f} hours</b> per refuel<br>"
            "Fuel saving: ₦%{customdata[0]:,.0f}/day · Loan: ₦%{customdata[1]:,.0f}/day"
            "<extra></extra>"
        ),
    ))
    # Direct label at the right end of each line. The two lines end close together,
    # so nudge the upper one up and the lower one down to keep the labels apart.
    last = d.iloc[-1]
    fig.add_annotation(
        x=last["earnings_per_hour"], y=last["break_even_queue_hours"],
        text=f"{vehicle_name}: {last['break_even_queue_hours']:.1f} hrs",
        xanchor="left", xshift=10, yshift=LABEL_SHIFT[vehicle_name], showarrow=False,
        font=dict(color=INK, size=12),
    )

fig.update_layout(
    title=dict(
        text="How long can a driver queue before CNG stops paying off?<br>"
             f"<sup>Mid scenario: petrol ₦{m['petrol_price']:,.0f}/L, CNG ₦{m['cng_price']:,.0f}/scm, "
             f"{m['interest_rate']:.1%} loan over {m['tenor_years']:g} years, {REFUELS_PER_DAY} refuel/day. "
             "Earnings per hour is a sensitivity range pending survey data.</sup>",
        x=0, xanchor="left",
    ),
    template="plotly_white",
    font=dict(family="Inter, Segoe UI, Arial, sans-serif", size=13, color=INK),
    xaxis=dict(
        title=dict(text="Driver net earnings (₦ per hour) — assumed range, pending survey", standoff=8),
        tickprefix="₦", tickformat=",.0f", dtick=500, showgrid=False,
        range=[EARNINGS_PER_HOUR[0] - 200, EARNINGS_PER_HOUR[-1] + 200],   # a little padding, no empty tick past 5,000
    ),
    yaxis=dict(
        title=dict(text="Break-even queue time (hours per refuel)", standoff=12),
        rangemode="tozero", gridcolor="#ececea", ticksuffix=" h",
    ),
    legend=dict(orientation="h", yanchor="bottom", y=1.0, x=0, xanchor="left"),
    hovermode="closest",
    margin=dict(t=130, l=80, r=170, b=140),
    height=620,
)

fig.add_annotation(
    text="Break-even = (fuel saving − loan cost) ÷ (refuels per day × earnings per hour). "
         "Mid inputs from data/raw/model_inputs.csv.<br>"
         "Sources (docs/sources.md): petrol & CNG price S11; loan S18; queue band S02, S04; "
         "petrol spend S21 (car), S19 (keke); conversion S40 (car), S43/S04 (keke).<br>"
         f"Petrol spend rescaled from ~₦{SPEND_REFERENCE_PETROL_PRICE:,}/L to ₦{m['petrol_price']:,.0f}/L "
         f"assuming the same litres per day. Keke conversion ₦{conv_keke['mid']:,.0f} = derived midpoint of "
         f"₦{conv_keke['low']:,.0f}–₦{conv_keke['high']:,.0f}. Working days {working_days['mid']:g}/yr (assumption).",
    xref="paper", yref="paper", x=0, y=-0.25, showarrow=False,
    xanchor="left", align="left", font=dict(size=11, color=INK_MUTED),
)

fig.show()

In [11]:
out_path = FIG_DIR / "break_even_queue.html"
fig.write_html(out_path, include_plotlyjs="cdn")   # cdn: small file, needs internet to render (see notebook 01)
print(f"Saved {out_path.relative_to(PROJECT_ROOT)}")

Saved outputs\figures\break_even_queue.html


## 9. Numbers for the write-up

Where each line crosses into the reported band: the earnings level at which break-even drops below 6 hours (the worst
reported queue), and the break-even at the two ends of the earnings range. These feed the explanation below.

In [12]:
for vehicle_name in vehicles:
    d = mid[mid["vehicle"] == vehicle_name].set_index("earnings_per_hour")["break_even_queue_hours"]
    inside = d[d <= BAND_HIGH]
    first_inside = inside.index.min() if len(inside) else None
    print(f"{vehicle_name}:")
    print(f"  break-even at ₦{d.index.min():,}/hr = {d.iloc[0]:.1f} hrs; at ₦{d.index.max():,}/hr = {d.iloc[-1]:.1f} hrs")
    print(f"  enters the reported band (<= {BAND_HIGH:g} hrs) from ₦{first_inside:,}/hr" if first_inside
          else "  stays above the reported band across the range")

Ride-hailing car:
  break-even at ₦1,000/hr = 19.2 hrs; at ₦5,000/hr = 3.8 hrs
  enters the reported band (<= 6 hrs) from ₦3,500/hr
Keke:
  break-even at ₦1,000/hr = 15.5 hrs; at ₦5,000/hr = 3.1 hrs
  enters the reported band (<= 6 hrs) from ₦3,000/hr


## 10. What the results mean

**For drivers earning less, queues are an annoyance, not a deal-breaker.** In the mid scenario, the ride-hailing car's
break-even stays above 6 hours (the worst reported queue) up to ₦3,000 per hour of earnings, and the keke's up to
₦2,500. A driver in that range could wait longer than any queue in our sources and still come out ahead on CNG.

**For drivers earning more, real queues can wipe out the gain.** Each line drops into the reported 1.5–6 hour band once
earnings pass about ₦3,000/hour (car from ₦3,500, keke from ₦3,000). At ₦5,000/hour the car breaks even at 3.8 hours and
the keke at 3.1. Both are *shorter* than the 4-hour mid queue, so at that earnings level a typical Abuja-style queue
makes conversion lose money. The better a driver's time is paid, the more each hour of queuing costs them.

**The keke has less room than the car.** Its kit is cheaper, but it burns less fuel, so there is less saving to absorb
queue time: its line sits below the car's at every earnings level.

**What it means for policy:** fuel prices alone make CNG attractive (notebook 01). In this model the binding
constraint is time at the pump. At the NESG estimate of 1.5 hours (S02), both vehicles pay off across the whole
earnings range; at 4–6 hours (S04), drivers earning more than about ₦3,000/hour sit at or past the tipping point. This points to station capacity and
queue times as the lever to test in the policy appraisal. The across-scenario table in section 7 shows the conclusion
holds as inputs move (at ₦3,000/hour: car 5.1–7.3 hours, keke 4.0–5.8 hours).

**Main assumptions, all to be replaced by survey data:**
- **Earnings per hour** (₦1,000–5,000) is a sensitivity range, not an estimate. It decides where on these lines real
  drivers sit.
- **1 refuel per day** and **efficiency ratio 1.0** (1 scm goes as far as 1 litre; calibrated for cars, C4).
- **Petrol spend rescaled assuming the same litres per day** as when reported at about ₦1,000/litre (S19, S21).
- **Keke mid conversion cost ₦375,000** is a derived midpoint, and **312 working days a year** is an assumption.
- **The conversion is loan-financed.** A driver who pays cash, or gets a free kit, has no loan cost and can tolerate
  slightly longer queues.